# 20 — CYP2D6 Residual Exclusion Screen (with a random-masking control)

**Cheap screen only: a single fold, three seeds per arm.** Does NOT run the full 25-fold CV,
does NOT build or evaluate an ensemble, does NOT prepare or send a submission.

## What this tests, and why

Notebook 06 found that excluding CYP2D6's worst-fitting training compounds (top 5% by pooled
out-of-fold `chemprop_chemeleoninit` residual) significantly improved CV ST-RAE for three of
four configs — `chemprop_chemeleoninit`'s own CYP2D6 CV ST-RAE moved **0.937 -> 0.897**
(p=1.6e-05). This is the largest untested CV-validated effect in the project (larger than
notebook 18's charge-descriptor screen, which found no effect at all) — **and it has never been
tested on the real board.** It was folded into notebook 07's tuning and then notebook 08's
ensembling, and what actually got submitted (NB10, NB12) were ensembles that collapsed on blind
data for reasons unrelated to exclusion. No submission has ever carried residual exclusion as an
isolated change.

## The unresolved confound this screen is designed to settle

Notebook 06 *also* found that masking CYP2D6 labels shifted the OTHER THREE isoforms'
predictions — Pearson r 0.92-0.95 against the notebook 05 baseline, not ~1.0 — even though
CYP1A2/CYP2C9/CYP3A4 labels were never touched. Training nondeterminism was ruled out
decisively (two identical back-to-back fits produced bit-identical checkpoints and predictions).
Masking *content* was ruled out too (residual and CI-width exclusion mask 74% different
compounds yet produce nearly identical divergence magnitude and per-isoform ordering). The
leading hypothesis, **never verified against chemprop's actual loss implementation**, is that
its multitask loss weights each task by how many valid labels it has in a batch — so masking
CYP2D6 labels changes CYP2D6's share of the combined loss and shifts the shared encoder's
gradients, moving the other three isoforms' outputs too.

**Consequence: a CYP2D6 improvement under exclusion currently cannot be attributed.** It could
be a genuine data-quality effect (the excluded compounds were bad labels or genuinely
unlearnable) or an implicit reweighting effect (any reduction in CYP2D6's label count shifts the
loss balance in a way that happens to help CYP2D6, regardless of which compounds are removed).

## Design: three arms, same fold, same three seeds, everything else identical

| arm      | what's masked |
|----------|----------------|
| BASELINE | nothing |
| RESIDUAL | notebook 06's residual-flagged CYP2D6 labels (this fold's training portion only) |
| RANDOM   | the SAME NUMBER of CYP2D6 labels, drawn uniformly at random from this fold's training CYP2D6 rows, **excluding** every compound on the residual list — disjoint from RESIDUAL's masked set, same random draw reused across all three training seeds |

If RESIDUAL improves CYP2D6 and RANDOM does not: **data quality.** If both improve by a similar
margin: **reweighting artifact** — a real finding about multitask training, meaning notebook
06's causal story needs rewriting. Both masked arms are also expected to shift
CYP1A2/CYP2C9/CYP3A4 predictions relative to BASELINE (notebook 06's r=0.92-0.95); if RESIDUAL
and RANDOM produce a *similar* cross-isoform shift, that's direct evidence for the reweighting
mechanism independent of whatever CYP2D6's own score does.

## Setup

In [1]:
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import importlib.metadata
import json
import os
from itertools import combinations

import numpy as np
import pandas as pd

from src.chemprop_screen import (
    build_predict_csv,
    build_training_csv,
    load_screen_population,
    run_chemprop_predict,
    run_chemprop_train,
    setup_logging,
    verify_predictions,
)
from src.cv_bootstrap import per_fold_bootstrap_seed
from src.vendor.openadmet_eval.config import ACTIVITY_METRICS, REGRESSION_ENDPOINTS
from src.vendor.openadmet_eval.evaluate_predictions import add_macro_endpoint, score_activity_predictions

METRIC_NAMES = [name for name, _ in ACTIVITY_METRICS]

print(f"python: {sys.version.split()[0]}")
for pkg in ["chemprop", "torch", "numpy", "pandas"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")
print(f"REGRESSION_ENDPOINTS: {REGRESSION_ENDPOINTS}")
print(f"METRIC_NAMES: {METRIC_NAMES}")

FOLDS_PATH = REPO_ROOT / "data" / "folds" / "cv_folds.csv"
CURATED_PATH = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
FLAGGED_PATH = REPO_ROOT / "outputs" / "06_outlier_check" / "flagged_compounds" / "criterion1_residual_flagged.csv"
STORED_05_SCORE_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "scores" / "chemprop_chemeleoninit__repeat0_fold0.csv"
STORED_05_MANIFEST_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "manifest.csv"

OUT = REPO_ROOT / "outputs" / "20_residual_exclusion_screen"
CHEMPROP_RUNS_DIR = OUT / "chemprop_runs"
PRED_DIR = OUT / "predictions"
SCORE_DIR = OUT / "scores"
LOG_DIR = REPO_ROOT / "logs" / "20_residual_exclusion_screen"
for d in [OUT, CHEMPROP_RUNS_DIR, PRED_DIR, SCORE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Same fold as notebook 06's own outlier check and notebook 18's screen: repeat_0, fold 0 -- not
# a new choice. This is what lets Part 1's baseline sanity check compare directly against
# outputs/05_cv_comparison's stored chemprop_chemeleoninit__repeat0_fold0 numbers, and lets Part
# 0 report notebook 06's own "59 of 75 flagged compounds land in this fold's training pool"
# figure as a direct, checkable cross-reference rather than a fresh, unrelated count.
REPEAT_COL = "repeat_0"
TEST_FOLD = 0
VAL_FRACTION = 0.15  # matches assign_screen_split's/05's/18's own precedent

# Same three training seeds as notebook 18's screen, for the same reason notebook 18 gave: not
# invented fresh here -- drawn from this project's own established
# np.random.SeedSequence(42).spawn(25) provenance (05's own manifest,
# scripts/generate_5x5_cv_manifest.py) for repeat_0's folds 0/1/2. SEEDS[0] is BIT-IDENTICAL to
# the seed 05's own chemprop_chemeleoninit__repeat0_fold0 run used, which is what makes the
# baseline sanity check below a clean apples-to-apples comparison rather than a different seed
# entirely.
SEEDS = [2684470948, 4091952314, 233227757]

# Separate, fixed, stated seed for the RANDOM arm's compound draw -- distinct from every training
# seed above and from cv_folds.csv's own build seed (42), so the random draw's own randomness is
# never confused with a training seed's randomness. Arbitrary (today's date, ISO YYYYMMDD), no
# provenance claim beyond "fixed and stated here." The SAME draw is reused across all three
# training seeds per the task's own instruction: the RANDOM arm should vary only by training
# seed, not by which compounds are masked.
RANDOM_MASK_SEED = 20260914

EPOCHS = 50
PATIENCE = 5
CHEMELEON_ARCH_ARGS = ["--from-foundation", "CHEMELEON", "--multi-hot-atom-featurizer-mode", "V2"]
CYP2D6_COL = "CYP2D6_pIC50_direct_inhibition"
OTHER_ISOFORM_COLS = [e for e in REGRESSION_ENDPOINTS if e != CYP2D6_COL]

print(f"\nfold: {REPEAT_COL}, test_fold={TEST_FOLD}")
print(f"seeds: {SEEDS}")
print(f"random mask seed: {RANDOM_MASK_SEED}")
print(f"epochs={EPOCHS}, patience={PATIENCE}")

python: 3.11.13
chemprop: 2.3.1
torch: 2.13.0
numpy: 1.26.4
pandas: 2.3.3
REGRESSION_ENDPOINTS: ['CYP1A2_pIC50_direct_inhibition', 'CYP2C9_pIC50_direct_inhibition', 'CYP2D6_pIC50_direct_inhibition', 'CYP3A4_pIC50_direct_inhibition']
METRIC_NAMES: ['ST-RAE', 'MAE', 'R2', 'Spearman_R', 'Kendall_Tau']

fold: repeat_0, test_fold=0
seeds: [2684470948, 4091952314, 233227757]
random mask seed: 20260914
epochs=50, patience=5


## Pre-registered interpretation (frozen before running anything)

Saved to `outputs/20_residual_exclusion_screen/prereg.json` **before** any training or scoring
cell below executes.

In [2]:
PREREG = {
    "notebook": "20_residual_exclusion_screen",
    "frozen_before": "any training or scoring in this notebook",
    "fold": {"repeat_col": REPEAT_COL, "test_fold": TEST_FOLD},
    "seeds": SEEDS,
    "random_mask_seed": RANDOM_MASK_SEED,
    "arms": {
        "baseline": "chemprop_chemeleoninit, no exclusion",
        "residual": (
            "identical, with notebook 06's residual-flagged CYP2D6 labels "
            "(criterion1_residual_flagged.csv) masked to NaN, restricted to this fold's "
            "training portion"
        ),
        "random": (
            "identical, with the SAME NUMBER of CYP2D6 labels masked to NaN, drawn uniformly "
            "at random from this fold's training CYP2D6 rows, excluding every compound on the "
            "residual list -- disjoint from the residual arm's masked set by construction, same "
            "draw reused across all three training seeds"
        ),
    },
    "background": (
        "Notebook 06 found residual-based CYP2D6 exclusion significantly improved CV ST-RAE for "
        "chemprop_chemeleoninit (0.937 -> 0.897, p=1.6e-05) alongside 2 tabular configs, with a "
        "real CI-width control showing a different pattern -- but never isolated whether the "
        "CYP2D6 gain is a genuine data-quality effect or an artifact of chemprop's multitask "
        "loss reweighting by valid-label count (06's own leading, never-verified hypothesis for "
        "why masking CYP2D6 labels shifted CYP1A2/CYP2C9/CYP3A4 predictions by r=0.92-0.95 "
        "against the 05 baseline, even though those isoforms' own labels were never touched). "
        "This notebook's RANDOM arm is designed to separate the two: it reduces CYP2D6's label "
        "count by the same amount as RESIDUAL, without removing the specific compounds RESIDUAL "
        "removes."
    ),
    "interpretation_rules": {
        "DATA_QUALITY": (
            "RESIDUAL improves CYP2D6 ST-RAE beyond the 3-seed spread AND RANDOM does not. The "
            "gain is attributable to which compounds were removed."
        ),
        "REWEIGHTING_ARTIFACT": (
            "RANDOM improves CYP2D6 by a similar margin to RESIDUAL. The gain comes from "
            "reducing CYP2D6's label count, not from the identity of the removed compounds -- a "
            "real finding about multitask training in its own right, meaning notebook 06's "
            "causal story needs rewriting."
        ),
        "NULL": (
            "Neither arm improves CYP2D6 beyond the 3-seed spread. Notebook 06's CV effect does "
            "not reproduce on this fold under this protocol; report plainly, recommend no "
            "further investment without new evidence."
        ),
        "MIXED": (
            "RANDOM improves but by materially less than RESIDUAL -- both mechanisms are "
            "contributing and cannot be cleanly separated by this design. State this rather "
            "than picking the more flattering reading."
        ),
    },
    "resolution_rule": (
        "a between-arm ST-RAE difference on CYP2D6 is 'resolved' only if it exceeds the larger "
        "of the two arms' own 3-seed standard deviation (ddof=1) -- otherwise call it "
        "unresolved, matching notebook 18's own precedent for this exact distinction."
    ),
    "reweighting_vs_mixed_magnitude_threshold": (
        "when both RESIDUAL and RANDOM are resolved-improve: RANDOM's improvement magnitude >= "
        "70% of RESIDUAL's improvement magnitude -> REWEIGHTING_ARTIFACT; below 70% -> MIXED. "
        "70% is an arbitrary, round, stated threshold, frozen here before running anything -- "
        "not derived from any result."
    ),
    "cross_isoform_expectation": (
        "Both masked arms (RESIDUAL and RANDOM) are expected to shift CYP1A2/CYP2C9/CYP3A4 "
        "predictions relative to BASELINE (notebook 06 observed r=0.92-0.95 for RESIDUAL vs. "
        "the 05 baseline, using the SAME seed per fold that comparison had available -- this "
        "notebook's own BASELINE arm, not 05's stored predictions, is the reference here, "
        "matched by training seed). If RANDOM produces a SIMILAR cross-isoform shift to "
        "RESIDUAL, that is direct evidence for the reweighting mechanism independent of any "
        "CYP2D6 score movement -- reported as such regardless of what the CYP2D6 result shows."
    ),
}

prereg_path = OUT / "prereg.json"
with open(prereg_path, "w") as f:
    json.dump(PREREG, f, indent=2)
print(f"saved {prereg_path}")
print(json.dumps(PREREG, indent=2))

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/prereg.json
{
  "notebook": "20_residual_exclusion_screen",
  "frozen_before": "any training or scoring in this notebook",
  "fold": {
    "repeat_col": "repeat_0",
    "test_fold": 0
  },
  "seeds": [
    2684470948,
    4091952314,
    233227757
  ],
  "random_mask_seed": 20260914,
  "arms": {
    "baseline": "chemprop_chemeleoninit, no exclusion",
    "residual": "identical, with notebook 06's residual-flagged CYP2D6 labels (criterion1_residual_flagged.csv) masked to NaN, restricted to this fold's training portion",
    "random": "identical, with the SAME NUMBER of CYP2D6 labels masked to NaN, drawn uniformly at random from this fold's training CYP2D6 rows, excluding every compound on the residual list -- disjoint from the residual arm's masked set by construction, same draw reused across all three training seeds"
  },
  "background": "Notebook 06 found residual-based CYP2D6 exclusion signif

## Part 0 — Setup and provenance (no recomputation)

`criterion1_residual_flagged.csv` is notebook 06's own already-computed flag set (top 5% of
CYP2D6-labeled compounds by pooled `chemprop_chemeleoninit` OOF `mean_abs_residual`, threshold
confirmed by the user before any CV-impact check, per that notebook's own record). **Not
recomputed here** — re-deriving the threshold would silently change the intervention under
test.

In [3]:
residual_flagged = pd.read_csv(FLAGGED_PATH)
print(f"loaded {FLAGGED_PATH.relative_to(REPO_ROOT)}")
print(f"file last modified: {pd.Timestamp(os.path.getmtime(FLAGGED_PATH), unit='s')}")
print(f"n compounds on notebook 06's criterion-1 (residual) flagged list: {len(residual_flagged)}")
print(residual_flagged.head())

loaded outputs/06_outlier_check/flagged_compounds/criterion1_residual_flagged.csv
file last modified: 2026-09-02 12:59:19.245929003
n compounds on notebook 06's criterion-1 (residual) flagged list: 75
                      inchikey Molecule_Name  mean_abs_residual  n_oof_obs
0  YCVRUZSBJRNHDW-VQIMIIECSA-N  OCNT-2315149           2.950343          5
1  SNCVEOXAGGEZHI-UHFFFAOYSA-N  OCNT-2308483           2.915033          5
2  OSXJAFIEDOREKR-UHFFFAOYSA-N  OCNT-2308589           2.855843          5
3  KFZXVMNBUMVKLN-UHFFFAOYSA-N  OCNT-2328764           2.833280          5
4  ACOCBYLGXYWEIH-UHFFFAOYSA-N  OCNT-2310884           2.791666          5


In [4]:
setup_log = setup_logging(LOG_DIR / "setup.log", "nb20_setup")

populations = {}
train_ik_sets = []
test_ik_sets = []
for seed in SEEDS:
    population = load_screen_population(
        FOLDS_PATH, CURATED_PATH, REPEAT_COL, TEST_FOLD, VAL_FRACTION, seed, setup_log
    )
    populations[seed] = population
    train_ik_sets.append(set(
        population.loc[population["screen_split"].isin(["screen_inner_train", "screen_inner_val"]), "inchikey"]
    ))
    test_ik_sets.append(set(population.loc[population["screen_split"] == "screen_test", "inchikey"]))

# The training-portion / held-out-portion compound SETS depend only on test_fold, not on seed --
# assign_screen_split's seed only controls how the training portion is split into
# screen_inner_train vs. screen_inner_val, not which compounds are in the training portion at
# all. Confirmed explicitly here rather than assumed, since everything downstream in Part 0
# treats "this fold's training portion" as one fixed set shared by all three seeds.
assert all(s == train_ik_sets[0] for s in train_ik_sets[1:]), \
    "training-portion compound set differs across seeds -- should be impossible"
assert all(s == test_ik_sets[0] for s in test_ik_sets[1:]), \
    "held-out compound set differs across seeds -- should be impossible"
TRAIN_IK = train_ik_sets[0]
TEST_IK = test_ik_sets[0]
assert TRAIN_IK.isdisjoint(TEST_IK), "training and held-out compound sets overlap -- should be impossible"
print(f"training portion (screen_inner_train + screen_inner_val, seed-invariant, confirmed across all 3 seeds): {len(TRAIN_IK)} compounds")
print(f"held-out portion (screen_test, seed-invariant, confirmed across all 3 seeds): {len(TEST_IK)} compounds")

2026-09-14 23:36:29,612 [INFO] loaded cv_folds.csv: (4905, 7)


2026-09-14 23:36:29,613 [INFO] loaded train_inhibition_curated.csv: (4905, 20)


2026-09-14 23:36:29,620 [INFO] assign_screen_split(repeat_col='repeat_0', test_fold=0, val_fraction=0.15, seed=2684470948) split counts: {'screen_inner_train': 3335, 'screen_test': 981, 'screen_inner_val': 589}


2026-09-14 23:36:29,622 [INFO] rows after SMILES/target join: 4905 (missing: 0)


2026-09-14 23:36:29,623 [INFO]   CYP1A2_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 975, 'screen_inner_val': 160, 'screen_test': 277}


...

2026-09-14 23:36:29,665 [INFO]   CYP1A2_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 965, 'screen_inner_val': 170, 'screen_test': 277}


2026-09-14 23:36:29,666 [INFO]   CYP2C9_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 864, 'screen_inner_val': 157, 'screen_test': 264}


2026-09-14 23:36:29,667 [INFO]   CYP2D6_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 1041, 'screen_inner_val': 157, 'screen_test': 295}


2026-09-14 23:36:29,668 [INFO]   CYP3A4_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 1587, 'screen_inner_val': 298, 'screen_test': 450}


training portion (screen_inner_train + screen_inner_val, seed-invariant, confirmed across all 3 seeds): 3924 compounds
held-out portion (screen_test, seed-invariant, confirmed across all 3 seeds): 981 compounds


### Which flagged compounds actually get masked, and the held-out-set safety assertion

In [5]:
flagged_ik = set(residual_flagged["inchikey"])
flagged_in_train = flagged_ik & TRAIN_IK
flagged_in_test = flagged_ik & TEST_IK
flagged_elsewhere = flagged_ik - TRAIN_IK - TEST_IK

print(f"of {len(flagged_ik)} globally-flagged compounds:")
print(f"  in THIS fold's training portion (these are the ones actually masked): {len(flagged_in_train)}")
print(f"  in THIS fold's held-out portion (screen_test, never masked): {len(flagged_in_test)}")
print(f"  in neither bucket (should be 0): {len(flagged_elsewhere)}")

if flagged_elsewhere:
    raise ValueError(f"{len(flagged_elsewhere)} flagged compounds not found in either bucket -- stopping.")

# The literal safety assertion: the set of compounds this notebook is actually about to mask
# (flagged_in_train) must have ZERO overlap with the held-out set -- masking held-out labels
# would corrupt the evaluation rather than the training. True by construction (flagged_in_train
# is an intersection with TRAIN_IK, which is already disjoint from TEST_IK, asserted above) --
# checked explicitly anyway rather than trusted.
assert flagged_in_train.isdisjoint(TEST_IK), \
    "a compound about to be masked is in the held-out set -- STOPPING, this would corrupt evaluation"
print("\nASSERT PASSED: the compounds about to be masked are fully disjoint from the held-out (screen_test) set.")

RESIDUAL_MASK_IK = flagged_in_train
N_MASKED = len(RESIDUAL_MASK_IK)
print(f"\nN_MASKED (RESIDUAL arm mask count, and the count the RANDOM arm must match): {N_MASKED}")

of 75 globally-flagged compounds:
  in THIS fold's training portion (these are the ones actually masked): 59
  in THIS fold's held-out portion (screen_test, never masked): 16
  in neither bucket (should be 0): 0

ASSERT PASSED: the compounds about to be masked are fully disjoint from the held-out (screen_test) set.

N_MASKED (RESIDUAL arm mask count, and the count the RANDOM arm must match): 59


### CYP2D6 training-label counts before/after masking

In [6]:
curated = pd.read_csv(CURATED_PATH)
train_labels = (
    pd.DataFrame({"inchikey": sorted(TRAIN_IK)})
    .merge(curated[["inchikey", CYP2D6_COL]], on="inchikey", how="left")
)
n_cyp2d6_train_before = int(train_labels[CYP2D6_COL].notna().sum())
n_cyp2d6_train_after = n_cyp2d6_train_before - N_MASKED

print(f"CYP2D6 training labels in this fold's training portion, before masking: {n_cyp2d6_train_before}")
print(f"CYP2D6 training labels in this fold's training portion, after masking {N_MASKED}: {n_cyp2d6_train_after}")

# every compound about to be masked must currently HAVE a real CYP2D6 label (it was flagged from
# an OOF residual, which requires a real label to compute in the first place) -- confirmed, not
# assumed.
masked_have_label = train_labels.loc[train_labels["inchikey"].isin(RESIDUAL_MASK_IK), CYP2D6_COL].notna().all()
assert masked_have_label, "at least one compound about to be masked has no CYP2D6 label to begin with -- stopping"
print("ASSERT PASSED: every compound about to be masked currently has a real CYP2D6 label.")

CYP2D6 training labels in this fold's training portion, before masking: 1198
CYP2D6 training labels in this fold's training portion, after masking 59: 1139
ASSERT PASSED: every compound about to be masked currently has a real CYP2D6 label.


### RANDOM arm's compound draw

Candidate pool: this fold's training-portion compounds with a real CYP2D6 label, **excluding
every compound on the full 75-compound residual-flagged list** (not just the 59 that happen to
land in this fold) — guarantees the two masked sets are disjoint regardless of fold membership,
and that a compound flagged as an outlier for a *different* fold's test set can't sneak into the
RANDOM draw here.

In [7]:
candidate_pool = train_labels.loc[
    train_labels[CYP2D6_COL].notna() & ~train_labels["inchikey"].isin(flagged_ik), "inchikey"
].to_numpy()
print(f"RANDOM arm candidate pool (training CYP2D6-labeled compounds, excluding all 75 flagged): {len(candidate_pool)}")

rng = np.random.default_rng(RANDOM_MASK_SEED)
RANDOM_MASK_IK = set(rng.choice(candidate_pool, size=N_MASKED, replace=False))
print(f"RANDOM arm draw: {len(RANDOM_MASK_IK)} compounds (seed={RANDOM_MASK_SEED}, matches N_MASKED={N_MASKED}: {len(RANDOM_MASK_IK) == N_MASKED})")

assert RANDOM_MASK_IK.isdisjoint(RESIDUAL_MASK_IK), "RANDOM and RESIDUAL masked sets overlap -- should be impossible by construction"
assert RANDOM_MASK_IK.isdisjoint(flagged_ik), "RANDOM draw includes a compound from the full 75-compound flagged list -- should be impossible"
assert RANDOM_MASK_IK.issubset(TRAIN_IK), "RANDOM draw includes a compound outside the training portion -- should be impossible"
print("\nASSERT PASSED: RANDOM and RESIDUAL masked sets are disjoint, and RANDOM never touches a flagged or held-out compound.")

RANDOM arm candidate pool (training CYP2D6-labeled compounds, excluding all 75 flagged): 1139
RANDOM arm draw: 59 compounds (seed=20260914, matches N_MASKED=59: True)

ASSERT PASSED: RANDOM and RESIDUAL masked sets are disjoint, and RANDOM never touches a flagged or held-out compound.


### The four byte-identical checks (reproducing notebook 06's own Section 7)

Applied here directly on the population DataFrame each arm's `train_input.csv` gets built from
(equivalent to checking the CSVs themselves, since `build_training_csv` derives the CSV from
exactly this DataFrame, unmodified) — at every seed, for both masked arms against that seed's
baseline population.

In [8]:
def apply_mask(population, mask_ik):
    masked = population.copy()
    hit = masked["inchikey"].isin(mask_ik)
    masked.loc[hit, CYP2D6_COL] = np.nan
    return masked


MASK_SETS = {"baseline": set(), "residual": RESIDUAL_MASK_IK, "random": RANDOM_MASK_IK}
masked_populations = {
    seed: {arm: apply_mask(populations[seed], MASK_SETS[arm]) for arm in MASK_SETS} for seed in SEEDS
}

print(f"{'seed':>12} {'arm':>10}  row_count  row_order  split_assign  other_3_cols  nan_pattern  cyp2d6_masked_correct")
for seed in SEEDS:
    base = populations[seed]
    for arm in ["residual", "random"]:
        masked = masked_populations[seed][arm]
        mask_ik = MASK_SETS[arm]

        check_row_count = len(masked) == len(base)
        check_row_order = (masked["inchikey"].to_numpy() == base["inchikey"].to_numpy()).all()
        check_split = (masked["screen_split"].to_numpy() == base["screen_split"].to_numpy()).all()
        check_other_cols = np.allclose(
            masked[OTHER_ISOFORM_COLS].to_numpy(dtype=float),
            base[OTHER_ISOFORM_COLS].to_numpy(dtype=float),
            equal_nan=True,
        )
        check_nan_pattern = (
            masked[OTHER_ISOFORM_COLS].isna().to_numpy() == base[OTHER_ISOFORM_COLS].isna().to_numpy()
        ).all()

        is_masked = masked["inchikey"].isin(mask_ik)
        check_cyp2d6_masked = masked.loc[is_masked, CYP2D6_COL].isna().all()
        check_cyp2d6_unmasked = np.allclose(
            masked.loc[~is_masked, CYP2D6_COL].to_numpy(dtype=float),
            base.loc[~is_masked, CYP2D6_COL].to_numpy(dtype=float),
            equal_nan=True,
        )
        check_cyp2d6_correct = check_cyp2d6_masked and check_cyp2d6_unmasked

        all_pass = all([
            check_row_count, check_row_order, check_split, check_other_cols,
            check_nan_pattern, check_cyp2d6_correct,
        ])
        print(f"{seed:>12} {arm:>10}  {check_row_count!s:>9}  {check_row_order!s:>9}  "
              f"{check_split!s:>12}  {check_other_cols!s:>12}  {check_nan_pattern!s:>11}  {check_cyp2d6_correct!s:>21}")
        if not all_pass:
            raise ValueError(f"seed={seed} arm={arm}: one of the byte-identical checks FAILED -- stopping.")

print("\nALL CHECKS PASS for both masked arms, at every seed: only CYP2D6 differs from baseline, exactly as intended.")

        seed        arm  row_count  row_order  split_assign  other_3_cols  nan_pattern  cyp2d6_masked_correct
  2684470948   residual       True       True          True          True         True                   True
  2684470948     random       True       True          True          True         True                   True
  4091952314   residual       True       True          True          True         True                   True
  4091952314     random       True       True          True          True         True                   True
   233227757   residual       True       True          True          True         True                   True
   233227757     random       True       True          True          True         True                   True

ALL CHECKS PASS for both masked arms, at every seed: only CYP2D6 differs from baseline, exactly as intended.


## Part 1 — Training

Three arms x three seeds = nine runs. BASELINE is retrained here (rather than reusing
`outputs/05_cv_comparison`'s stored numbers) so all three arms share fold, seeds, environment
and Chemprop version end-to-end. Masking is applied purely at the population-DataFrame level
(Part 0, above) *before* calling `src/chemprop_screen.py`'s existing `build_training_csv` /
`build_predict_csv` / `run_chemprop_train` / `run_chemprop_predict` — all reused completely
unmodified, no local variants needed (unlike notebook 18's charge screen, which needed a
descriptor-column variant). This is a true single-variable change: everything about how a run is
launched is identical across arms except which population DataFrame it's handed.

In [9]:
def run_one_arm_seed(arm, seed):
    tag = f"{arm}__seed{seed}"
    cached_pred_path = PRED_DIR / f"{tag}.csv"
    population = masked_populations[seed][arm]

    if cached_pred_path.exists():
        # Matches this project's own is_done()-checkpointing convention
        # (scripts/run_5x5_cv_comparison.py) -- lets this notebook be re-executed cheaply
        # without redoing real training.
        cached = pd.read_csv(cached_pred_path)
        expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])
        if set(cached["Molecule_Name"]) == expected_names and len(cached) == len(expected_names):
            print(f"  [cached] {tag}: reusing {cached_pred_path}")
            return cached
        print(f"  [cache mismatch] {tag}: cached file doesn't match this population -- retraining.")

    run_dir = CHEMPROP_RUNS_DIR / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    logger = setup_logging(LOG_DIR / f"{tag}.log", f"nb20_{tag}")

    train_csv = run_dir / "train_input.csv"
    build_training_csv(population, REGRESSION_ENDPOINTS, train_csv, logger, require_all_targets=False)
    predict_csv = build_predict_csv(population, run_dir, logger)

    run_chemprop_train(train_csv, REGRESSION_ENDPOINTS, run_dir, logger, CHEMELEON_ARCH_ARGS, EPOCHS, PATIENCE, seed)

    raw_pred_csv = run_dir / "raw_predictions.csv"
    run_chemprop_predict(run_dir / "model_0", predict_csv, raw_pred_csv, logger)

    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])
    verify_predictions(raw_pred_csv, expected_names, REGRESSION_ENDPOINTS, logger)

    pred_df = pd.read_csv(raw_pred_csv)
    pred_df.to_csv(cached_pred_path, index=False)
    return pred_df


RUN_ORDER = [(arm, seed) for arm in ["baseline", "residual", "random"] for seed in SEEDS]
print("run order:", RUN_ORDER)
print(f"total runs: {len(RUN_ORDER)}")

run order: [('baseline', 2684470948), ('baseline', 4091952314), ('baseline', 233227757), ('residual', 2684470948), ('residual', 4091952314), ('residual', 233227757), ('random', 2684470948), ('random', 4091952314), ('random', 233227757)]
total runs: 9


In [10]:
raw_predictions = {}
timings = {}

arm0, seed0 = RUN_ORDER[0]
print(f"=== FIRST RUN: arm={arm0}, seed={seed0} -- timing this one explicitly before launching the rest ===")
t0 = time.time()
raw_predictions[(arm0, seed0)] = run_one_arm_seed(arm0, seed0)
t1 = time.time()
timings[(arm0, seed0)] = t1 - t0
print(f"\nFIRST RUN ({arm0}, seed={seed0}) took {t1 - t0:.1f}s ({(t1 - t0) / 60:.2f} min)")
print(f"rough budget estimate for the remaining {len(RUN_ORDER) - 1} runs at this pace: "
      f"~{(len(RUN_ORDER) - 1) * (t1 - t0) / 60:.1f} min (actual per-run time varies with early stopping)")

=== FIRST RUN: arm=baseline, seed=2684470948 -- timing this one explicitly before launching the rest ===
  [cached] baseline__seed2684470948: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/predictions/baseline__seed2684470948.csv

FIRST RUN (baseline, seed=2684470948) took 0.0s (0.00 min)
rough budget estimate for the remaining 8 runs at this pace: ~0.0 min (actual per-run time varies with early stopping)


In [11]:
for arm, seed in RUN_ORDER[1:]:
    print(f"\n=== arm={arm}, seed={seed} ===")
    t0 = time.time()
    raw_predictions[(arm, seed)] = run_one_arm_seed(arm, seed)
    t1 = time.time()
    timings[(arm, seed)] = t1 - t0
    print(f"({arm}, seed={seed}) took {t1 - t0:.1f}s ({(t1 - t0) / 60:.2f} min)")

total_min = sum(timings.values()) / 60
print(f"\ntotal training+predict time across all {len(RUN_ORDER)} runs: {total_min:.2f} min")
for (arm, seed), secs in timings.items():
    print(f"  {arm} seed={seed}: {secs:.1f}s")


=== arm=baseline, seed=4091952314 ===
  [cached] baseline__seed4091952314: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/predictions/baseline__seed4091952314.csv
(baseline, seed=4091952314) took 0.0s (0.00 min)

=== arm=baseline, seed=233227757 ===
  [cached] baseline__seed233227757: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/predictions/baseline__seed233227757.csv
(baseline, seed=233227757) took 0.0s (0.00 min)

=== arm=residual, seed=2684470948 ===
  [cached] residual__seed2684470948: reusing /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/predictions/residual__seed2684470948.csv
(residual, seed=2684470948) took 0.0s (0.00 min)

=== arm=residual, seed=4091952314 ===
2026-09-14 23:36:29,733 [INFO] train_input.csv: pooled-train rows before label filter=3924, after=3924 (require_all_targets=False, targets=['CYP1A2_pIC50_direct_inhibition', 'CYP2C9_

2026-09-14 23:36:29,742 [INFO] wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/chemprop_runs/residual__seed4091952314/train_input.csv (3924 rows, chemprop_split counts: {'train': 3335, 'val': 589})


2026-09-14 23:36:29,744 [INFO] wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/chemprop_runs/residual__seed4091952314/predict_input.csv (981 screen_test compounds)


2026-09-14 23:36:29,745 [INFO] running: chemprop train -i /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/chemprop_runs/residual__seed4091952314/train_input.csv -s canonical_smiles --target-columns CYP1A2_pIC50_direct_inhibition CYP2C9_pIC50_direct_inhibition CYP2D6_pIC50_direct_inhibition CYP3A4_pIC50_direct_inhibition --splits-column chemprop_split -t regression --from-foundation CHEMELEON --multi-hot-atom-featurizer-mode V2 --epochs 50 --patience 5 --data-seed 4091952314 --pytorch-seed 4091952314 --accelerator cpu --devices 1 -o /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/chemprop_runs/residual__seed4091952314


2026-09-14 23:36:29,746 [INFO]   epoch 0 done: train_loss=0.9156, val_loss=0.7653


...

2026-09-15 00:13:40,877 [INFO] Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 0:00:04 • 0:00:00 3.97it/s


2026-09-15 00:13:40,886 [INFO] 2026-09-15T00:13:40 - INFO:chemprop.cli.predict - Predictions saved to '/Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/chemprop_runs/random__seed233227757/raw_predictions.csv'


2026-09-15 00:13:41,411 [INFO] rows after reading raw_predictions.csv back from disk: 981


2026-09-15 00:13:41,412 [INFO] NaN values in raw_predictions.csv target column(s): 0


(random, seed=233227757) took 333.5s (5.56 min)

total training+predict time across all 9 runs: 37.19 min
  baseline seed=2684470948: 0.0s
  baseline seed=4091952314: 0.0s
  baseline seed=233227757: 0.0s
  residual seed=2684470948: 0.0s
  residual seed=4091952314: 430.7s
  residual seed=233227757: 287.9s
  random seed=2684470948: 697.7s
  random seed=4091952314: 481.8s
  random seed=233227757: 333.5s


## Part 2 — Question A: does CYP2D6 improve, and which arm?

Held-out predictions scored with the vendored, unmodified evaluator
(`src/vendor/openadmet_eval/`), same call pattern as notebook 18 / `scripts/run_5x5_cv_comparison.py`
(each run's own training seed reused as its bootstrap seed).

In [12]:
def score_run(arm, seed, pred_df):
    ground_truth = curated[curated["inchikey"].isin(pred_df["inchikey"])].copy()
    with per_fold_bootstrap_seed(seed):
        scored = score_activity_predictions(pred_df, ground_truth, REGRESSION_ENDPOINTS)
        scored = add_macro_endpoint(scored, REGRESSION_ENDPOINTS, ACTIVITY_METRICS)
    scored["arm"] = arm
    scored["seed"] = seed
    scored.to_csv(SCORE_DIR / f"{arm}__seed{seed}.csv", index=False)
    return scored


scores = {}
for (arm, seed), pred_df in raw_predictions.items():
    scores[(arm, seed)] = score_run(arm, seed, pred_df)
    print(f"scored {arm} seed={seed}: {len(scores[(arm, seed)])} bootstrap rows")

point_rows = []
for (arm, seed), scored in scores.items():
    pe = scored.groupby("Endpoint")[METRIC_NAMES].mean()
    for endpoint in pe.index:
        row = {"arm": arm, "seed": seed, "Endpoint": endpoint}
        row.update(pe.loc[endpoint].to_dict())
        point_rows.append(row)
point_df = pd.DataFrame(point_rows)
point_df.to_csv(OUT / "point_estimates.csv", index=False)
print(f"\nsaved {OUT / 'point_estimates.csv'} ({len(point_df)} rows)")
print(point_df.round(4).to_string(index=False))

2026-09-15 00:13:41.426 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-15 00:13:41.428 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-09-15 00:13:41.428 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-15 00:13:41.428 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


2026-09-15 00:13:42.130 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP2C9_pIC50_direct_inhibition


...

2026-09-15 00:14:06.922 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric MAE across bootstrap iterations


2026-09-15 00:14:06.923 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-15 00:14:06.923 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-15 00:14:06.923 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


scored random seed=233227757: 5000 bootstrap rows

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/point_estimates.csv (45 rows)
     arm       seed                       Endpoint  ST-RAE    MAE     R2  Spearman_R  Kendall_Tau
baseline 2684470948 CYP1A2_pIC50_direct_inhibition  0.7884 0.6590 0.2949      0.5275       0.3714
baseline 2684470948 CYP2C9_pIC50_direct_inhibition  0.7113 0.5278 0.3609      0.5834       0.4112
baseline 2684470948 CYP2D6_pIC50_direct_inhibition  0.9580 0.5985 0.1650      0.4170       0.2830
baseline 2684470948 CYP3A4_pIC50_direct_inhibition  0.4910 0.5297 0.6054      0.7946       0.5973
baseline 2684470948                             MA  0.7372 0.5788 0.3565      0.5806       0.4157
baseline 4091952314 CYP1A2_pIC50_direct_inhibition  0.8159 0.6635 0.2706      0.5167       0.3657
baseline 4091952314 CYP2C9_pIC50_direct_inhibition  0.7312 0.5359 0.3375      0.5520       0.3913
baseline 4091952314 CYP2D6_pIC50_direct_in

### Baseline sanity check: retrained baseline (seed matching 05's own) vs. 05's stored numbers

`SEEDS[0]` (2684470948) is bit-identical to the seed `outputs/05_cv_comparison`'s own
`chemprop_chemeleoninit__repeat0_fold0` run used for this exact fold — same data, same
architecture, same environment. Notebook 18's equivalent check matched to 0.0000 ST-RAE; this
one is expected to be close but not necessarily bit-identical (this notebook's own BASELINE arm
loads/masks the population independently, and dict/set iteration order elsewhere in this
notebook is not guaranteed to match 05's original run byte-for-byte).

In [13]:
stored_05 = pd.read_csv(STORED_05_SCORE_PATH)
stored_pe = stored_05.groupby("Endpoint")[METRIC_NAMES].mean()

retrained_pe = point_df[(point_df["arm"] == "baseline") & (point_df["seed"] == seed0)].set_index("Endpoint")

print(f"{'Endpoint':35s} {'stored 05 ST-RAE':>18s} {'retrained ST-RAE':>18s} {'abs diff':>10s}")
max_diff = 0.0
for endpoint in REGRESSION_ENDPOINTS:
    stored_val = stored_pe.loc[endpoint, "ST-RAE"]
    retrained_val = retrained_pe.loc[endpoint, "ST-RAE"]
    diff = abs(stored_val - retrained_val)
    max_diff = max(max_diff, diff)
    print(f"{endpoint:35s} {stored_val:18.4f} {retrained_val:18.4f} {diff:10.4f}")

DISCREPANCY_FLAG_THRESHOLD = 0.03  # matches notebook 18's own flag threshold
print(f"\nmax abs ST-RAE difference: {max_diff:.4f} (flag threshold: {DISCREPANCY_FLAG_THRESHOLD})")
if max_diff > DISCREPANCY_FLAG_THRESHOLD:
    print("FLAG: retrained baseline differs from 05's stored numbers by more than the flag "
          "threshold on at least one isoform -- investigate before trusting the RESIDUAL/RANDOM "
          "comparisons below.")
else:
    print("OK: retrained baseline matches 05's stored numbers closely -- proceeding.")

Endpoint                              stored 05 ST-RAE   retrained ST-RAE   abs diff
CYP1A2_pIC50_direct_inhibition                  0.7884             0.7884     0.0000
CYP2C9_pIC50_direct_inhibition                  0.7113             0.7113     0.0000
CYP2D6_pIC50_direct_inhibition                  0.9580             0.9580     0.0000
CYP3A4_pIC50_direct_inhibition                  0.4910             0.4910     0.0000

max abs ST-RAE difference: 0.0000 (flag threshold: 0.03)
OK: retrained baseline matches 05's stored numbers closely -- proceeding.


### Full metric summary: ST-RAE, MAE, R2, Spearman -- per isoform, per arm, mean +/- (ddof=1) spread across the 3 seeds

In [14]:
summary_rows = []
for endpoint in REGRESSION_ENDPOINTS:
    iso = endpoint.split("_")[0]
    for arm in ["baseline", "residual", "random"]:
        sub = point_df[(point_df["Endpoint"] == endpoint) & (point_df["arm"] == arm)]
        row = {"isoform": iso, "arm": arm, "n_seeds": len(sub)}
        for metric in ["ST-RAE", "MAE", "R2", "Spearman_R"]:
            vals = sub[metric].to_numpy()
            row[f"{metric}_mean"] = vals.mean()
            row[f"{metric}_std"] = vals.std(ddof=1)
        summary_rows.append(row)
metrics_summary = pd.DataFrame(summary_rows)
metrics_summary.to_csv(OUT / "metrics_summary.csv", index=False)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
print(metrics_summary.round(4).to_string(index=False))

isoform      arm  n_seeds  ST-RAE_mean  ST-RAE_std  MAE_mean  MAE_std  R2_mean  R2_std  Spearman_R_mean  Spearman_R_std
 CYP1A2 baseline        3       0.7995      0.0145    0.6663   0.0091   0.2704  0.0245           0.5141          0.0150
 CYP1A2 residual        3       0.7939      0.0020    0.6618   0.0112   0.2845  0.0223           0.5261          0.0300
 CYP1A2   random        3       0.7925      0.0193    0.6642   0.0171   0.2758  0.0409           0.5264          0.0422
 CYP2C9 baseline        3       0.7142      0.0158    0.5315   0.0041   0.3524  0.0130           0.5767          0.0221
 CYP2C9 residual        3       0.7393      0.0342    0.5440   0.0039   0.3327  0.0186           0.5631          0.0134
 CYP2C9   random        3       0.7251      0.0385    0.5372   0.0044   0.3473  0.0169           0.5761          0.0096
 CYP2D6 baseline        3       0.9599      0.0626    0.6023   0.0275   0.1251  0.0545           0.4056          0.0220
 CYP2D6 residual        3       0.9362  

### CYP2D6 verdict: RESIDUAL and RANDOM reported separately against BASELINE -- no cross-isoform averaging

A between-arm difference is called **resolved** only if it exceeds the larger of the two arms'
own 3-seed standard deviation (frozen in `prereg.json` above) — otherwise it's **unresolved**,
not "small".

In [15]:
def st_rae_verdict(endpoint, arm):
    iso = endpoint.split("_")[0]
    base_row = metrics_summary[(metrics_summary["isoform"] == iso) & (metrics_summary["arm"] == "baseline")].iloc[0]
    arm_row = metrics_summary[(metrics_summary["isoform"] == iso) & (metrics_summary["arm"] == arm)].iloc[0]
    diff = arm_row["ST-RAE_mean"] - base_row["ST-RAE_mean"]  # negative = arm improves (lower ST-RAE)
    combined_spread = max(base_row["ST-RAE_std"], arm_row["ST-RAE_std"])
    resolved = abs(diff) > combined_spread
    if not resolved:
        verdict = "unresolved (diff smaller than within-arm seed spread)"
    elif diff < 0:
        verdict = f"{arm.upper()} improves"
    else:
        verdict = f"{arm.upper()} loses"
    return {
        "isoform": iso, "arm": arm,
        "baseline_mean": base_row["ST-RAE_mean"], "baseline_std": base_row["ST-RAE_std"],
        "arm_mean": arm_row["ST-RAE_mean"], "arm_std": arm_row["ST-RAE_std"],
        "diff": diff, "combined_seed_spread": combined_spread, "resolved": bool(resolved), "verdict": verdict,
    }


print("=== CYP2D6 (primary target of this screen) ===\n")
residual_verdict = st_rae_verdict(CYP2D6_COL, "residual")
random_verdict = st_rae_verdict(CYP2D6_COL, "random")
print(f"BASELINE: {residual_verdict['baseline_mean']:.4f} +/- {residual_verdict['baseline_std']:.4f}")
print(f"RESIDUAL: {residual_verdict['arm_mean']:.4f} +/- {residual_verdict['arm_std']:.4f}  "
      f"diff={residual_verdict['diff']:+.4f}, combined_seed_spread={residual_verdict['combined_seed_spread']:.4f} "
      f"-> {residual_verdict['verdict']}")
print(f"RANDOM:   {random_verdict['arm_mean']:.4f} +/- {random_verdict['arm_std']:.4f}  "
      f"diff={random_verdict['diff']:+.4f}, combined_seed_spread={random_verdict['combined_seed_spread']:.4f} "
      f"-> {random_verdict['verdict']}")

print("\n=== other isoforms (labels never touched in any arm, reported for completeness, NOT averaged with CYP2D6) ===\n")
other_verdicts = {}
for endpoint in OTHER_ISOFORM_COLS:
    iso = endpoint.split("_")[0]
    rv = st_rae_verdict(endpoint, "residual")
    ranv = st_rae_verdict(endpoint, "random")
    other_verdicts[iso] = {"residual": rv, "random": ranv}
    print(f"{iso}: BASELINE={rv['baseline_mean']:.4f}+/-{rv['baseline_std']:.4f}  "
          f"| RESIDUAL diff={rv['diff']:+.4f} -> {rv['verdict']}  "
          f"| RANDOM diff={ranv['diff']:+.4f} -> {ranv['verdict']}")

=== CYP2D6 (primary target of this screen) ===

BASELINE: 0.9599 +/- 0.0626
RESIDUAL: 0.9362 +/- 0.0249  diff=-0.0238, combined_seed_spread=0.0626 -> unresolved (diff smaller than within-arm seed spread)
RANDOM:   0.9639 +/- 0.0601  diff=+0.0040, combined_seed_spread=0.0626 -> unresolved (diff smaller than within-arm seed spread)

=== other isoforms (labels never touched in any arm, reported for completeness, NOT averaged with CYP2D6) ===

CYP1A2: BASELINE=0.7995+/-0.0145  | RESIDUAL diff=-0.0057 -> unresolved (diff smaller than within-arm seed spread)  | RANDOM diff=-0.0070 -> unresolved (diff smaller than within-arm seed spread)
CYP2C9: BASELINE=0.7142+/-0.0158  | RESIDUAL diff=+0.0251 -> unresolved (diff smaller than within-arm seed spread)  | RANDOM diff=+0.0109 -> unresolved (diff smaller than within-arm seed spread)
CYP3A4: BASELINE=0.5223+/-0.0313  | RESIDUAL diff=-0.0349 -> RESIDUAL improves  | RANDOM diff=-0.0017 -> unresolved (diff smaller than within-arm seed spread)


## Part 3 — Question B: the cross-isoform divergence

For CYP1A2, CYP2C9 and CYP3A4 — whose labels were never touched in any arm — the Pearson
correlation of each masked arm's held-out predictions against **this notebook's own BASELINE**
predictions, matched by training seed (same seed => same architecture init/data-loader
behaviour, isolating the masking effect).

In [16]:
cross_iso_rows = []
for arm in ["residual", "random"]:
    for seed in SEEDS:
        base_pred = raw_predictions[("baseline", seed)].set_index("inchikey")
        arm_pred = raw_predictions[(arm, seed)].set_index("inchikey")
        merged = base_pred.join(arm_pred, lsuffix="_base", rsuffix="_arm", how="inner")
        assert len(merged) == len(base_pred) == len(arm_pred), \
            f"{arm} seed={seed}: held-out compound sets don't match 1:1"
        for endpoint in OTHER_ISOFORM_COLS:
            iso = endpoint.split("_")[0]
            r = merged[f"{endpoint}_base"].corr(merged[f"{endpoint}_arm"])
            mean_abs_diff = (merged[f"{endpoint}_arm"] - merged[f"{endpoint}_base"]).abs().mean()
            cross_iso_rows.append({
                "arm": arm, "seed": seed, "isoform": iso,
                "pearson_r_vs_baseline": r, "mean_abs_diff_vs_baseline": mean_abs_diff,
            })
cross_iso_df = pd.DataFrame(cross_iso_rows)
cross_iso_df.to_csv(OUT / "cross_isoform_divergence_vs_baseline.csv", index=False)
print(cross_iso_df.round(4).to_string(index=False))

     arm       seed isoform  pearson_r_vs_baseline  mean_abs_diff_vs_baseline
residual 2684470948  CYP1A2                 0.9140                     0.2208
residual 2684470948  CYP2C9                 0.9476                     0.1662
residual 2684470948  CYP3A4                 0.9490                     0.2111
residual 4091952314  CYP1A2                 0.9524                     0.1678
residual 4091952314  CYP2C9                 0.9704                     0.1341
residual 4091952314  CYP3A4                 0.9701                     0.1682
residual  233227757  CYP1A2                 0.9870                     0.1047
residual  233227757  CYP2C9                 0.9857                     0.0891
residual  233227757  CYP3A4                 0.9867                     0.1290
  random 2684470948  CYP1A2                 0.9135                     0.2197
  random 2684470948  CYP2C9                 0.9476                     0.1644
  random 2684470948  CYP3A4                 0.9427              

### Reference point: BASELINE seed vs. BASELINE seed (the noise floor)

How much two identical-recipe models differ from training-seed alone — the floor any
masking-induced divergence must clear to be attributable to masking rather than ordinary
run-to-run noise. Notebook 18 found same-recipe error correlations of 0.984-0.990 (a related but
not identical quantity — that notebook compared against a pooled-OOF reference column, not a
literal same-recipe rerun); this notebook has three literal BASELINE reruns at different seeds,
so the floor below is computed directly.

In [17]:
baseline_pair_rows = []
for s1, s2 in combinations(SEEDS, 2):
    p1 = raw_predictions[("baseline", s1)].set_index("inchikey")
    p2 = raw_predictions[("baseline", s2)].set_index("inchikey")
    merged = p1.join(p2, lsuffix="_s1", rsuffix="_s2", how="inner")
    assert len(merged) == len(p1) == len(p2), f"baseline seed pair {s1},{s2}: held-out compound sets don't match 1:1"
    for endpoint in REGRESSION_ENDPOINTS:
        iso = endpoint.split("_")[0]
        r = merged[f"{endpoint}_s1"].corr(merged[f"{endpoint}_s2"])
        mean_abs_diff = (merged[f"{endpoint}_s1"] - merged[f"{endpoint}_s2"]).abs().mean()
        baseline_pair_rows.append({
            "seed_pair": f"{s1}_vs_{s2}", "isoform": iso, "pearson_r": r, "mean_abs_diff": mean_abs_diff,
        })
baseline_pair_df = pd.DataFrame(baseline_pair_rows)
baseline_pair_df.to_csv(OUT / "baseline_seed_vs_seed_reference.csv", index=False)
print(baseline_pair_df.round(4).to_string(index=False))

baseline_floor_by_iso = baseline_pair_df.groupby("isoform")["pearson_r"].mean()
print("\nbaseline seed-vs-seed reference (mean r across the 3 pairs), per isoform -- the noise floor:")
print(baseline_floor_by_iso.round(4).to_string())

               seed_pair isoform  pearson_r  mean_abs_diff
2684470948_vs_4091952314  CYP1A2     0.8806         0.2606
2684470948_vs_4091952314  CYP2C9     0.9238         0.1877
2684470948_vs_4091952314  CYP2D6     0.7773         0.2611
2684470948_vs_4091952314  CYP3A4     0.9168         0.2677
 2684470948_vs_233227757  CYP1A2     0.9010         0.3201
 2684470948_vs_233227757  CYP2C9     0.9177         0.1957
 2684470948_vs_233227757  CYP2D6     0.8728         0.2026
 2684470948_vs_233227757  CYP3A4     0.9126         0.2751
 4091952314_vs_233227757  CYP1A2     0.8330         0.3671
 4091952314_vs_233227757  CYP2C9     0.8879         0.2127
 4091952314_vs_233227757  CYP2D6     0.7999         0.2718
 4091952314_vs_233227757  CYP3A4     0.9013         0.2884

baseline seed-vs-seed reference (mean r across the 3 pairs), per isoform -- the noise floor:
isoform
CYP1A2    0.8715
CYP2C9    0.9098
CYP2D6    0.8167
CYP3A4    0.9102


### RESIDUAL vs. RANDOM: similar cross-isoform divergence is direct evidence for the reweighting mechanism

In [18]:
summary_3_3 = []
for endpoint in OTHER_ISOFORM_COLS:
    iso = endpoint.split("_")[0]
    floor_r = baseline_floor_by_iso.loc[iso]
    res_r = cross_iso_df.loc[(cross_iso_df["arm"] == "residual") & (cross_iso_df["isoform"] == iso), "pearson_r_vs_baseline"].mean()
    rand_r = cross_iso_df.loc[(cross_iso_df["arm"] == "random") & (cross_iso_df["isoform"] == iso), "pearson_r_vs_baseline"].mean()
    gap = res_r - rand_r
    summary_3_3.append({
        "isoform": iso, "baseline_seed_floor_r": floor_r,
        "residual_vs_baseline_mean_r": res_r, "random_vs_baseline_mean_r": rand_r,
        "residual_minus_random_r": gap,
    })
    print(f"{iso}: baseline-seed floor r={floor_r:.4f}  |  RESIDUAL vs baseline mean r={res_r:.4f}  "
          f"|  RANDOM vs baseline mean r={rand_r:.4f}  |  RESIDUAL-RANDOM gap={gap:+.4f}")
summary_3_3_df = pd.DataFrame(summary_3_3)
summary_3_3_df.to_csv(OUT / "cross_isoform_summary.csv", index=False)

print("\nFor reference, notebook 06's own residual-vs-CI-width comparison (a different pair of "
      "disjoint masked sets, same underlying question) found gaps of 0.005-0.013 between two "
      "criteria masking 74% different compounds -- both diverging from the 05 baseline by "
      "similar, criterion-independent amounts. The gaps above are read against that precedent, "
      "not an arbitrary cutoff invented fresh here.")

CYP1A2: baseline-seed floor r=0.8715  |  RESIDUAL vs baseline mean r=0.9511  |  RANDOM vs baseline mean r=0.9543  |  RESIDUAL-RANDOM gap=-0.0032
CYP2C9: baseline-seed floor r=0.9098  |  RESIDUAL vs baseline mean r=0.9679  |  RANDOM vs baseline mean r=0.9699  |  RESIDUAL-RANDOM gap=-0.0020
CYP3A4: baseline-seed floor r=0.9102  |  RESIDUAL vs baseline mean r=0.9686  |  RANDOM vs baseline mean r=0.9680  |  RESIDUAL-RANDOM gap=+0.0006

For reference, notebook 06's own residual-vs-CI-width comparison (a different pair of disjoint masked sets, same underlying question) found gaps of 0.005-0.013 between two criteria masking 74% different compounds -- both diverging from the 05 baseline by similar, criterion-independent amounts. The gaps above are read against that precedent, not an arbitrary cutoff invented fresh here.


## Part 4 — Verdict (report, do not act)

Applying the criteria frozen in `prereg.json` above, without softening in either direction.

In [19]:
base_resolved_improves = lambda v: v["resolved"] and v["diff"] < 0

residual_improves = base_resolved_improves(residual_verdict)
random_improves = base_resolved_improves(random_verdict)

REWEIGHTING_MAGNITUDE_THRESHOLD = 0.70  # frozen in prereg.json above

if residual_improves and not random_improves:
    final_call = "DATA QUALITY"
    final_reasoning = (
        "RESIDUAL is a resolved improvement on CYP2D6 and RANDOM is not (unresolved or a loss) "
        "-- the gain is attributable to which compounds were removed, not merely to reducing "
        "CYP2D6's label count."
    )
elif residual_improves and random_improves:
    magnitude_ratio = abs(random_verdict["diff"]) / abs(residual_verdict["diff"])
    if magnitude_ratio >= REWEIGHTING_MAGNITUDE_THRESHOLD:
        final_call = "REWEIGHTING ARTIFACT"
        final_reasoning = (
            f"Both arms are resolved improvements on CYP2D6, and RANDOM's magnitude "
            f"({magnitude_ratio:.1%} of RESIDUAL's) clears the frozen {REWEIGHTING_MAGNITUDE_THRESHOLD:.0%} "
            "threshold -- the gain comes from reducing CYP2D6's label count, not from the "
            "identity of the removed compounds. Notebook 06's causal story for this effect "
            "needs rewriting."
        )
    else:
        final_call = "MIXED"
        final_reasoning = (
            f"Both arms are resolved improvements on CYP2D6, but RANDOM's magnitude "
            f"({magnitude_ratio:.1%} of RESIDUAL's) falls short of the frozen "
            f"{REWEIGHTING_MAGNITUDE_THRESHOLD:.0%} threshold -- both mechanisms appear to be "
            "contributing and this design cannot cleanly separate them."
        )
elif not residual_improves and not random_improves:
    final_call = "NULL"
    final_reasoning = (
        "Neither arm is a resolved improvement on CYP2D6 -- notebook 06's CV effect does not "
        "reproduce on this fold under this protocol."
    )
else:
    final_call = "UNEXPECTED"
    final_reasoning = (
        "RANDOM is a resolved improvement but RESIDUAL is not -- not one of the four "
        "pre-registered outcomes. Reported literally rather than force-fit into one of them."
    )

print(f"FINAL VERDICT: {final_call}")
print(f"\n{final_reasoning}")

print("\n--- cross-isoform (reweighting-mechanism) evidence, independent of the CYP2D6 result above ---")
for row in summary_3_3:
    print(f"  {row['isoform']}: RESIDUAL r={row['residual_vs_baseline_mean_r']:.4f}, "
          f"RANDOM r={row['random_vs_baseline_mean_r']:.4f} (gap={row['residual_minus_random_r']:+.4f}, "
          f"noise floor={row['baseline_seed_floor_r']:.4f})")

full_25_fold_candidate = final_call == "DATA QUALITY"
print(f"\nCYP2D6 candidate for the full 25-fold CV comparison on this screen's evidence: "
      f"{full_25_fold_candidate} (NOT run in this task)")
print("No submission recommendation is made by this notebook.")

verdict = {
    "final_call": final_call,
    "final_reasoning": final_reasoning,
    "residual_verdict": residual_verdict,
    "random_verdict": random_verdict,
    "cross_isoform_summary": summary_3_3,
    "full_25_fold_cv_candidate": bool(full_25_fold_candidate),
    "submission_recommendation": "none -- this notebook is a report-only screen",
}
verdict_path = OUT / "verdict.json"
with open(verdict_path, "w") as f:
    json.dump(verdict, f, indent=2, default=float)
print(f"\nsaved {verdict_path}")

FINAL VERDICT: NULL

Neither arm is a resolved improvement on CYP2D6 -- notebook 06's CV effect does not reproduce on this fold under this protocol.

--- cross-isoform (reweighting-mechanism) evidence, independent of the CYP2D6 result above ---
  CYP1A2: RESIDUAL r=0.9511, RANDOM r=0.9543 (gap=-0.0032, noise floor=0.8715)
  CYP2C9: RESIDUAL r=0.9679, RANDOM r=0.9699 (gap=-0.0020, noise floor=0.9098)
  CYP3A4: RESIDUAL r=0.9686, RANDOM r=0.9680 (gap=+0.0006, noise floor=0.9102)

CYP2D6 candidate for the full 25-fold CV comparison on this screen's evidence: False (NOT run in this task)
No submission recommendation is made by this notebook.

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/20_residual_exclusion_screen/verdict.json
